In [ ]:

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

import keras_tuner as kt

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("Churn Modeling.csv")
df.head()
df.shape
df.info()
df.describe().T
print(df.isnull().sum())


### Key Insights from EDA

#### Dataset Overview
- The dataset contains **10,000 customer records** and **14 features**.
- No missing values were found, so no data imputation was required.

#### Churn Analysis
- Approximately **20.37%** of customers have churned, while **79.63%** remain with the bank.
- The target variable is imbalanced, making metrics like F1-Score and ROC-AUC important.

#### Customer Characteristics
- The average customer age is **39 years**.
- The average credit score is **651**, indicating moderate credit quality.
- Customers have an average tenure of **5 years** with the bank.

#### Balance and Salary
- A significant number of customers have **zero account balance**.
- Balance shows high variability and potential skewness.
- Estimated Salary is relatively evenly distributed across customers.

#### Product and Membership Insights
- Most customers use **1–2 banking products**.
- Around **70.5%** of customers own a credit card.
- About **51.5%** of customers are active members.

In [ ]:
df['Exited'].value_counts()

In [ ]:
plt.figure(figsize=(6,4))

sns.countplot(
    x='Exited',
    data=df,
    palette='Set2'
)

plt.title("Customer Churn Distribution")
plt.show()


#### Customer Churn Distribution

- The dataset is **imbalanced**, with significantly more customers retained than churned.
- Approximately **79.6%** of customers stayed with the bank, while **20.4%** of customers exited.
- This indicates that customer churn is a relatively less frequent event.
- Due to class imbalance, evaluation metrics such as **Precision, Recall, F1-Score, and ROC-AUC** are important in addition to Accuracy.
- The imbalance may lead the model to favor the majority class, so careful model evaluation is required.

#### Business Insight
- Around **1 in every 5 customers** leaves the bank.
- Identifying customers at risk of churn can help the bank implement targeted retention strategies and reduce customer loss.


In [ ]:

plt.figure(figsize=(8,5))

sns.countplot(
    x='Geography',
    hue='Exited',
    data=df
)

plt.show()


#### Customer Churn by Geography

- France has the highest number of customers, followed by Spain and Germany.
- Germany shows a relatively higher proportion of churned customers compared to France and Spain.
- Spain has the lowest customer churn among the three countries.
- Customers from Germany appear more likely to leave the bank than customers from other regions.

#### Business Insight
- Customer retention strategies should focus more on the German customer segment.
- Understanding regional factors affecting churn can help design targeted marketing and retention campaigns.


In [ ]:

sns.countplot(
    x='Gender',
    hue='Exited',
    data=df
)

plt.show()


#### Customer Churn by Gender

- The dataset contains more male customers than female customers.
- Female customers show a higher churn count compared to male customers.
- Male customers have a higher retention rate, with more customers staying with the bank.
- Gender appears to have an influence on customer churn behavior.

#### Business Insight
- Female customers may require additional retention strategies and personalized engagement programs.
- Gender can be considered an important feature for churn prediction models.


In [ ]:
num_cols = [
    'CreditScore',
    'Age',
    'Balance',
    'EstimatedSalary',
    'Tenure',
    'NumOfProducts'
]

for col in num_cols:

    plt.figure(figsize=(6,3))

    sns.histplot(df[col], kde=True)

    plt.title(col)

    plt.show()


#### Insights from Numerical Feature Distributions

##### Credit Score
- Credit scores are approximately normally distributed around the average score of 650.
- Most customers have moderate credit ratings, with relatively few customers at the extreme ends.

##### Age
- The age distribution is slightly right-skewed.
- Most customers fall between 30 and 45 years of age.
- A few older customers create potential outliers in the dataset.

##### Balance
- The balance distribution is highly skewed.
- A significant number of customers maintain zero account balance.
- Presence of high-balance customers creates a long right tail.

##### Estimated Salary
- Estimated salary appears relatively evenly distributed across customers.
- No major concentration is observed in specific salary ranges.

##### Tenure
- Customer tenure is fairly uniform between 0 and 10 years.
- Customers have varying levels of relationship duration with the bank.

##### Number of Products
- Most customers use 1 or 2 banking products.
- Very few customers use 3 or 4 products.
- Product usage is concentrated among lower values.

##### Overall Observation
- Age and Balance show noticeable skewness and potential outliers.
- Balance is likely to require transformation before model training.
- Feature scaling is necessary to improve ANN performance.

In [ ]:

plt.figure(figsize=(12,8))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    cmap='coolwarm'
)

plt.show()


#### Correlation Heatmap

- Most variables show **weak correlations**, indicating that each feature contributes unique information to the model.
- **Age** has the strongest positive correlation with churn (**0.29**), suggesting that older customers are more likely to leave the bank.
- **IsActiveMember** has a moderate negative correlation with churn (**-0.16**), indicating that active customers are less likely to churn.
- **Balance** shows a positive correlation with churn (**0.12**), meaning customers with higher balances tend to have a higher probability of leaving.
- **NumOfProducts** has a slight negative correlation with churn (**-0.05**), suggesting that customers using more products are generally less likely to churn.
- **CreditScore**, **Tenure**, **HasCrCard**, and **EstimatedSalary** have very weak correlations with churn, indicating limited direct influence individually.
- A moderate negative correlation (**-0.30**) exists between **Balance** and **NumOfProducts**, suggesting customers with higher balances often use fewer products.

##### Business Insight
- Customer churn is primarily influenced by **Age**, **Account Balance**, and **Customer Activity Status**.
- Increasing customer engagement and encouraging active usage of banking services may help reduce churn.


In [ ]:

skew_df = pd.DataFrame({
    'Feature': num_cols,
    'Skewness': [skew(df[col]) for col in num_cols]
})

skew_df


#### Skewness Analysis

- **CreditScore (-0.07)** is nearly symmetric with no significant skewness.
- **Age (1.01)** is positively skewed, indicating the presence of more younger customers and a few older customers acting as outliers.
- **Balance (-0.14)** is approximately symmetric and does not require major transformation.
- **EstimatedSalary (0.002)** is almost perfectly normally distributed.
- **Tenure (0.01)** is evenly distributed across customers with negligible skewness.
- **NumOfProducts (0.75)** shows moderate positive skewness, indicating that most customers use fewer banking products.

##### Conclusion
- Moderate skewness mainly observed in **Age** and **NumOfProducts**.
- Skewness removal not required.
- Most features are close to a normal distribution and are suitable for model building after feature scaling.


In [ ]:

for col in num_cols:

    plt.figure(figsize=(6,3))

    sns.boxplot(x=df[col])

    plt.title(col)

    plt.show()


#### Outlier Analysis
- **Age** contains the most prominent outliers and should be monitored during preprocessing.
- **Credit Score** has a few lower-end outliers but remains relatively stable.
- **Balance** exhibits high variability, indicating differences in customer account holdings.
- Outlier treatment using the **IQR method** can improve model robustness and generalization.
- The salary distribution is fairly symmetric with a median around **100,000**.
- Most customers use **1 to 2 banking products**.
- A small number of customers use **4 products**, appearing as outliers.
- Product usage is concentrated at lower values, indicating limited cross-selling.
- **EstimatedSalary** and **Tenure** do not contain significant outliers.
- **NumOfProducts** contains a few upper-end outliers but they may represent genuine customer behavior.
- Overall, the dataset has limited outlier issues in these features and requires minimal treatment.

In [ ]:
""" def cap_outliers(df,col):

    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3-q1

    lower = q1 - 1.5*iqr
    upper = q3 + 1.5*iqr

    df[col] = np.where(df[col] > upper, upper, df[col])

    df[col] = np.where(df[col] < lower, lower, df[col])

    return df


for col in num_cols:
    df = cap_outliers(df,col)
"""

In [ ]:
df.drop(['RowNumber',
         'CustomerId',
         'Surname'],
         axis=1,
         inplace=True)


#### Feature Selection

- **RowNumber** was removed because it is only a sequential identifier and does not provide useful information for predicting customer churn.
- **CustomerId** was dropped as it is a unique identifier and has no relationship with customer behavior.
- **Surname** was removed because customer names do not contribute meaningful predictive information and can introduce unnecessary noise.
- Removing these columns helps reduce dimensionality and prevents the model from learning irrelevant patterns.
- The remaining features contain customer demographic, financial, and behavioral information that are more relevant for churn prediction.
- Identifier and non-informative columns were removed to improve model efficiency and focus on meaningful predictors of customer churn.


In [ ]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['Geography'] = df['Geography'].map({'France': 0, 'Germany': 1,'Spain':2})


#### Encoding

- The **Gender** feature was converted into numerical format, where **Male = 1** and **Female = 0**, making it suitable for machine learning algorithms.
- The **Geography** feature was transformed using **One-Hot Encoding** to convert categorical country values into separate binary columns.
- Categorical features were successfully converted into numerical representations.
- The transformed dataset is now ready for feature scaling and model training.

In [ ]:

X = df.drop('Exited', axis=1)
y = df['Exited']

In [ ]:

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [ ]:

ann = Sequential()

ann.add(Dense(
    128,
    activation='relu',
    input_shape=(X_train.shape[1],)
))

ann.add(Dropout(0.3))

ann.add(Dense(
    64,
    activation='relu'
))

ann.add(Dropout(0.2))

ann.add(Dense(
    32,
    activation='relu'
))

ann.add(Dense(
    16,
    activation='relu'
))

ann.add(Dense(
    1,
    activation='sigmoid'
))

In [ ]:
ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)
checkpoint = ModelCheckpoint(
    'Best_ANN.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

history = ann.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[
        early_stop,
        checkpoint
    ]
)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.legend(
    ['Train','Validation']
)

plt.title("Accuracy Curve")

plt.show()

In [ ]:
print("Best accuracy", max(history.history["accuracy"]))
print("Best validation accuracy", max(history.history["val_accuracy"]))

In [ ]:

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.legend(
    ['Train','Validation']
)

plt.title("Loss Curve")

plt.show()


In [ ]:
ann_prob = ann.predict(X_test)

ann_pred = (
    ann_prob > 0.5
).astype(int)


print(
    classification_report(
        y_test,
        ann_pred
    )
)

In [ ]:
def build_model(hp):

    model = Sequential()

    model.add(
        Dense(
            hp.Int(
                'units1',
                min_value=32,
                max_value=256,
                step=32),
            activation=hp.Choice(
                'act1',
                ['relu','tanh','elu']),
            input_dim=X_train.shape[1])
    )

    model.add(
        Dropout(
            hp.Float(
                'dropout1',
                0.0,
                0.5,
                step=0.1))
    )

    for i in range(
        hp.Int(
            'hidden_layers',
            1,
            4)
    ):

        model.add(
            Dense(
                hp.Int(
                    f'units_{i}',
                    min_value=32,
                    max_value=256,
                    step=32),
                activation=hp.Choice(
                    f'act_{i}',
                    ['relu','tanh','elu'])
            )
        )

    model.add(Dense(1,
                    activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice(
            'optimizer',
            ['adam','rmsprop','nadam']),
        loss='binary_crossentropy',
        metrics=['accuracy'])

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    overwrite=True,
    directory='churn_tuning',
    project_name='ann'
)

In [ ]:

tuner.search(
    X_train,
    y_train,
    epochs=30,
    validation_split=0.2
)

In [ ]:

best_model = tuner.get_best_models(
    num_models=1
)[0]



In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)
checkpoint = ModelCheckpoint(
    'Best_ANN_Tuned.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

besttunedmodel = best_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[
        early_stop,
        checkpoint
    ]
)

In [ ]:
print(max(besttunedmodel.history["accuracy"]))
print(max(besttunedmodel.history["val_accuracy"]))


In [ ]:
best_prob = best_model.predict(X_test)

best_pred = (
    best_prob > 0.5
).astype(int)

best_model.save('Best_ANN_tuned.keras')

import joblib

joblib.dump(scaler,"featurescaler.pkl")

print(best_prob)


In [ ]:

results = pd.DataFrame({

'Model':
[
'ANN',
'Tuned ANN'
],

'Accuracy':
[
accuracy_score(y_test,ann_pred),
accuracy_score(y_test,best_pred)
],

'Precision':
[
precision_score(y_test,ann_pred),
precision_score(y_test,best_pred)
],

'Recall':
[
recall_score(y_test,ann_pred),
recall_score(y_test,best_pred)
],

'F1 Score':
[
f1_score(y_test,ann_pred),
f1_score(y_test,best_pred)
]

})

results.sort_values(
    by='Accuracy',
    ascending=False
)


| Model | Accuracy | Precision | Recall | F1 Score |
|---------|---------:|----------:|--------:|---------:|
| **Tuned ANN** | **0.8635** | 0.7445 | **0.5012** | **0.5991** |
| ANN | 0.8605 | **0.7689** | 0.4496 | 0.5674 |


### Details

- The **Tuned ANN** achieved a slightly higher accuracy (**86.35%**) compared to the ANN (**86.05%**).
- Recall improved from **44.96%** to **50.12%**, indicating better identification of positive cases.
- Although precision decreased slightly (**76.89% → 74.45%**), the improvement in recall led to a higher **F1 Score** (**0.5991 vs 0.5674**).
- Overall, the **Tuned ANN** provides a better balance between precision and recall, making it the preferred model.
``


In [ ]:

cm = confusion_matrix(
    y_test,
    best_pred
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")

plt.show()


In [ ]:

auc = roc_auc_score(
    y_test,
    best_prob
)

fpr,tpr,_ = roc_curve(
    y_test,
    best_prob
)

plt.figure(figsize=(8,5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    '--'
)

plt.xlabel("FPR")
plt.ylabel("TPR")

plt.legend()

plt.title("ROC Curve")

plt.show()



### ROC Curve Insights

- The ROC curve lies well above the diagonal baseline, indicating that the model performs significantly better than random guessing.
- The model achieved an **AUC (Area Under the Curve) of 0.858**, which reflects strong classification capability.
- An AUC value of **0.858** means the model has an **85.8% probability** of correctly distinguishing between positive and negative classes.
- The curve rises steeply at lower False Positive Rates (FPR), showing that the model can achieve high True Positive Rates (TPR) while keeping false alarms low.
- The model demonstrates a good balance between sensitivity (recall) and specificity across different classification thresholds.
- Since the AUC value is closer to **1.0** than **0.5**, the model exhibits strong discriminative power and reliable predictive performance.
- Overall, the ROC curve confirms that the pretrained CNN is an effective model for distinguishing between **Accident** and **Non-Accident** images.
